# Gaussian Process Regression using LOGO cross-validation on subsets of data

### Step 1: Setup, data loading, and descriptor definition (LOGO CV)

In this notebook, we evaluate a **leave-one-carbonyl-out (LOGO)** cross-validation strategy to quantify how well a Gaussian Process Regression (GPR) model can extrapolate to **previously unseen carbonyl substrates**. In contrast to the screening notebook, this version is restricted to **one fixed descriptor** (`dg_ch3`) and **one fixed kernel** (**Matern 1.5 + White**) - the best performing model identified - and is designed to test how predictive performance changes when only a fraction of the available training data is used.

For each specified training fraction, we:

1. **Construct a dataset** containing the four reaction parameters together with the  
   selected carbonyl descriptor `dg_ch3`.
2. **Use `carbonyl_nr` as the LOGO grouping variable**, so that all reactions for  
   one carbonyl substrate are kept together in the held-out test fold.
3. **Subsample the training portion of each LOGO fold**, while leaving the  
   held-out carbonyl fold unchanged. This allows us to evaluate model quality as a  
   function of training-set size.
4. **Fit a Gaussian Process Regression (GPR) model** using the fixed covariance  
   kernel **Matern 1.5 + White**. Kernel hyperparameters are optimized during  
   fitting via bounded maximum log-likelihood estimation using a capped  
   L-BFGS-B optimizer.

This cell sets up the environment, loads and preprocesses the datasets,  
defines the single descriptor and model used throughout the notebook,  
and prepares the data for the subsequent LOGO analysis.

In [21]:
import os
import re
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel as C
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from scipy.optimize import fmin_l_bfgs_b


# --------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------
RANDOM_STATE = 42
DATASET_FILE = "dataset_r2filtered_ml.xlsx"
PARAM_FILE = "carbonyl_parameters.xlsx"
SINGLE_DESCRIPTOR = "dg_ch3"
KERNEL_LABEL = "Matern_1.5 + White"
MODEL_NAME = f"GPR[{KERNEL_LABEL}]"
TRAIN_SIZE_FRACTIONS = np.array([0.01, 0.02, 0.03, 0.04, 0.05, 0.10, 0.15, 0.20, 0.30]) # This is the fraction of data used for training in each LOGO fold
N_SUBSAMPLE_REPEATS = 10 # This is the number of repeats when sampling with a fraction of the data
MIN_TRAIN_ROWS = 8 # This is the minimal number of rows used for training
SUMMARY_CSV = "results_logo_training_fraction_dg_ch3_matern15.csv"
PER_FOLD_CSV = "results_logo_each_fraction_dg_ch3_matern15.csv"
BASE_PLOTS_DIR = "plots_logo_training_fraction_dg_ch3_matern15"
SHOW_PLOTS = False

Path(BASE_PLOTS_DIR).mkdir(exist_ok=True, parents=True)

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
})

# Suppress non-critical GPR warnings
warnings.filterwarnings("ignore", message=".*divide by zero encountered in matmul.*")
warnings.filterwarnings("ignore", message=".*overflow encountered in matmul.*")
warnings.filterwarnings("ignore", message=".*invalid value encountered in matmul.*")

# --------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred) -> float:
    return float(mean_absolute_error(y_true, y_pred))

def sanitize(name: str) -> str:
    """Safe filename stem."""
    return re.sub(r"[^A-Za-z0-9_\.%-]+", "_", name)

def parity_plot_and_save_logo_global(
    model_name: str,
    y_cv,
    y_cv_pred,
    ylabel: str,
    outdir: str,
    q2: float,
    mae_cv: float,
    suffix: str | None = None,
) -> None:
    """Global LOGO parity plot for pooled out-of-fold predictions."""
    y_cv = np.asarray(y_cv)
    y_cv_pred = np.asarray(y_cv_pred)

    y_all = np.concatenate([y_cv, y_cv_pred])
    y_min, y_max = float(y_all.min()), float(y_all.max())

    fig, ax = plt.subplots(figsize=(6, 5.5))
    ax.scatter(y_cv, y_cv_pred, s=28, alpha=0.8, label="LOGO predictions")
    ax.plot([y_min, y_max], [y_min, y_max], "k--", lw=1.4, label="y = x")

    title = f"{model_name} — Q²(LOGO)={q2:.3f}, MAE(cv)={mae_cv:.3f}"
    if suffix:
        title += f" [{suffix}]"
    ax.set_title(title)
    ax.set_xlabel(f"Actual {ylabel}")
    ax.set_ylabel(f"Predicted {ylabel}")
    ax.legend()
    plt.tight_layout()

    os.makedirs(outdir, exist_ok=True)
    filename = f"parity_LOGO_global_{sanitize(model_name)}"
    if suffix:
        filename += f"_{sanitize(suffix)}"
    filename += ".png"
    fig.savefig(
        os.path.join(outdir, filename),
        dpi=300,
        bbox_inches="tight",
        transparent=True,
    )
    if SHOW_PLOTS:
        plt.show()
    plt.close(fig)

def make_gpr_kernel():
    """Single GPR kernel requested for the training-size analysis."""
    C_bounds = (1e-3, 1e5)
    L_bounds = (1e-3, 1e3)
    N_bounds = (1e-8, 1e2)
    return C(1.0, C_bounds) * (
        Matern(length_scale=1.0, nu=1.5, length_scale_bounds=L_bounds)
        + WhiteKernel(noise_level=1.0, noise_level_bounds=N_bounds)
    )

def _optimizer_with_cap(obj_func, initial_theta, bounds, maxfun=60, maxiter=60):
    theta_opt, func_min, _ = fmin_l_bfgs_b(
        obj_func, initial_theta, bounds=bounds,
        maxfun=maxfun, maxiter=maxiter,
    )
    return theta_opt, float(func_min)

def get_model_template():
    return GaussianProcessRegressor(
        kernel=make_gpr_kernel(),
        normalize_y=True,
        random_state=RANDOM_STATE,
        n_restarts_optimizer=1,
        alpha=1e-8,
        optimizer=_optimizer_with_cap,
    )

def subsample_training_rows(X_train_full, y_train_full, fraction, seed):
    n_full = X_train_full.shape[0]
    n_take = max(MIN_TRAIN_ROWS, int(math.ceil(fraction * n_full)))
    n_take = min(n_take, n_full)

    if n_take >= n_full:
        return X_train_full, y_train_full

    rng = np.random.RandomState(seed)
    sel = rng.choice(n_full, size=n_take, replace=False)
    return X_train_full[sel], y_train_full[sel]

# --------------------------------------------------------------------
# Data loading and descriptor preparation
# --------------------------------------------------------------------
df_ds = pd.read_excel(DATASET_FILE)
df_params = pd.read_excel(PARAM_FILE)

reaction_cols_all = ["ha_pka", "pH_corr", "[nucleophile]", "[ha]tot", "k_obs"]
for c in reaction_cols_all:
    if c in df_ds.columns:
        df_ds[c] = pd.to_numeric(df_ds[c], errors="coerce")

df_ds = df_ds.dropna(subset=["k_obs"]).copy()
df_ds = df_ds[df_ds["k_obs"] > 0].copy()
df_ds["lg_k_obs"] = np.log10(df_ds["k_obs"])

reaction_cols = [c for c in ["ha_pka", "pH_corr", "[nucleophile]", "[ha]tot"] if c in df_ds.columns]
join_key = "carbonyl_nr"

if SINGLE_DESCRIPTOR not in df_params.columns:
    raise ValueError(
        f"Descriptor '{SINGLE_DESCRIPTOR}' was not found in {PARAM_FILE}. "
        f"Available columns include: {list(df_params.columns)}"
    )

sub_params = df_params[[join_key, SINGLE_DESCRIPTOR]].copy()
sub_params[SINGLE_DESCRIPTOR] = pd.to_numeric(sub_params[SINGLE_DESCRIPTOR], errors="coerce")
merged = df_ds.merge(sub_params, on=join_key, how="inner")

feat_cols = reaction_cols + [SINGLE_DESCRIPTOR]
merged = merged.dropna(subset=feat_cols + ["lg_k_obs"]).copy()

if merged.shape[0] < 25:
    raise ValueError(
        f"Only {merged.shape[0]} usable rows remain for descriptor '{SINGLE_DESCRIPTOR}'."
    )

X = merged[feat_cols].values
y = merged["lg_k_obs"].values
groups = merged[join_key].values
target_name = "lg_k_obs"
logo = LeaveOneGroupOut()
model_template = get_model_template()

print(f"Loaded dataset with {len(y)} usable reactions.")
print(f"Unique held-out groups (carbonyl_nr): {np.unique(groups).size}")
print(f"Using descriptor: {SINGLE_DESCRIPTOR}")
print(f"Using model: {MODEL_NAME}")
print(f"Training fractions: {TRAIN_SIZE_FRACTIONS.tolist()}")
print(f"Random subsampling repeats per fraction: {N_SUBSAMPLE_REPEATS}")

Loaded dataset with 6889 usable reactions.
Unique held-out groups (carbonyl_nr): 10
Using descriptor: dg_ch3
Using model: GPR[Matern_1.5 + White]
Training fractions: [0.01, 0.02, 0.03, 0.04, 0.05, 0.1, 0.15, 0.2, 0.3]
Random subsampling repeats per fraction: 10


### Step 2: LOGO training across training-set fractions, pooled Q²(LOGO), and parity plots

In this step, we perform **leave-one-carbonyl-out (LOGO) cross-validation** to
evaluate how predictive performance depends on the **size of the training set**.

The analysis is carried out for a **single descriptor** (`dg_ch3`) and a **fixed GPR model**
(**Matern 1.5 + White kernel**), while systematically varying the fraction of data used
for training.

The workflow is:

1. **Define LOGO folds**  
   The dataset is split using `carbonyl_nr` as the grouping variable.  
   In each fold, all reactions for one carbonyl substrate are held out as the test set,  
   while all remaining substrates form the training pool.

2. **Subsample the training data**  
   For each specified value in `TRAIN_SIZE_FRACTIONS`, only a fraction of the available  
   training rows in each LOGO fold is randomly selected.  
   - The held-out test fold remains unchanged.  
   - Subsampling is repeated multiple times (`N_SUBSAMPLE_REPEATS`) with different  
     random seeds to obtain more stable estimates.

3. **Train and evaluate the GPR model per fold**  
   - Inputs are standardized using `StandardScaler` fitted **only on the subsampled  
     training data**.  
   - A Gaussian Process Regression model with a **Matern 1.5 + White kernel** is trained.  
   - Kernel hyperparameters are optimized via bounded maximum log-likelihood estimation  
     using a capped L-BFGS-B optimizer with one restart.  

   For each fold, we compute:
   - R², MAE, and RMSE for both training and test sets  
   - The number of samples used for training (`n_train_used`) and available in the full  
     training pool (`n_train_full`)

4. **Compute pooled LOGO performance (Q²)**  
   After completing all LOGO folds for a given fraction and repeat, the predictions on  
   the held-out substrates are concatenated.  
   From these pooled out-of-fold predictions we compute:  
   - **Q²(LOGO)** (cross-validated R²),  
   - **MAE(cv)**, and  
   - **RMSE(cv)**.  

   A **global LOGO parity plot** is generated for each training fraction and repeat.

5. **Aggregate and save results**  
   - A **per-fold CSV** containing detailed metrics for every fold, fraction, and repeat:  
     `results_logo_each_fraction_dg_ch3_matern15.csv`  
   - A **summary CSV** containing pooled Q²(LOGO), MAE(cv), and RMSE(cv) for each  
     training fraction and repeat:  
     `results_logo_training_fraction_dg_ch3_matern15.csv`

All parity plots are saved under:  
`plots_logo_training_fraction_dg_ch3_matern15/<descriptor>/`

In [2]:
LOGO_RESULTS_CSV = SUMMARY_CSV
PLOTS_DIR = os.path.join(BASE_PLOTS_DIR, sanitize(SINGLE_DESCRIPTOR))
os.makedirs(PLOTS_DIR, exist_ok=True)

fold_rows = []
summary_rows = []

print(
    f"Running LOGO training-size study for descriptor '{SINGLE_DESCRIPTOR}' with model {MODEL_NAME}\n"
)

for fraction in TRAIN_SIZE_FRACTIONS:
    print("\n" + "#" * 88)
    print(f"Training fraction: {fraction:.0%}")
    print("#" * 88)

    repeat_q2 = []
    repeat_mae = []
    repeat_rmse = []

    for repeat_idx in range(N_SUBSAMPLE_REPEATS):
        print(f"  Repeat {repeat_idx + 1}/{N_SUBSAMPLE_REPEATS}")

        y_true_all = []
        y_pred_all = []

        for fold_idx, (tr_idx, te_idx) in enumerate(logo.split(X, y, groups), start=1):
            sub_name_arr = np.unique(groups[te_idx])
            assert len(sub_name_arr) == 1
            sub_name = str(sub_name_arr[0])

            X_train_full, X_test = X[tr_idx], X[te_idx]
            y_train_full, y_test = y[tr_idx], y[te_idx]

            seed = RANDOM_STATE + repeat_idx * 1000 + fold_idx
            X_train, y_train = subsample_training_rows(
                X_train_full, y_train_full, fraction=fraction, seed=seed
            )

            scaler = StandardScaler().fit(X_train)
            X_train_use = scaler.transform(X_train)
            X_test_use = scaler.transform(X_test)

            model = clone(model_template)
            model.fit(X_train_use, y_train)

            y_train_pred = model.predict(X_train_use)
            y_test_pred = model.predict(X_test_use)

            r2_train = r2_score(y_train, y_train_pred)
            r2_test = r2_score(y_test, y_test_pred)
            mae_train = mae(y_train, y_train_pred)
            mae_test = mae(y_test, y_test_pred)
            rmse_train = rmse(y_train, y_train_pred)
            rmse_test = rmse(y_test, y_test_pred)

            fold_rows.append({
                "Descriptor": SINGLE_DESCRIPTOR,
                "Model": MODEL_NAME,
                "Train_Fraction": float(fraction),
                "Train_Percent": int(round(fraction * 100)),
                "Repeat": int(repeat_idx + 1),
                "Fold": int(fold_idx),
                "Heldout_Substrate": sub_name,
                "R2(train)": r2_train,
                "MAE(train)": mae_train,
                "RMSE(train)": rmse_train,
                "R2(test)": r2_test,
                "MAE(test)": mae_test,
                "RMSE(test)": rmse_test,
                "n_train_full": int(len(y_train_full)),
                "n_train_used": int(len(y_train)),
                "n_test": int(len(y_test)),
            })

            y_true_all.append(y_test)
            y_pred_all.append(y_test_pred)

        y_true_cv = np.concatenate(y_true_all)
        y_pred_cv = np.concatenate(y_pred_all)
        q2 = r2_score(y_true_cv, y_pred_cv)
        mae_cv = mae(y_true_cv, y_pred_cv)
        rmse_cv = rmse(y_true_cv, y_pred_cv)

        repeat_q2.append(q2)
        repeat_mae.append(mae_cv)
        repeat_rmse.append(rmse_cv)

        suffix = f"fraction={int(round(fraction * 100)):02d}pct_repeat={repeat_idx + 1}"
        parity_plot_and_save_logo_global(
            model_name=MODEL_NAME,
            y_cv=y_true_cv,
            y_cv_pred=y_pred_cv,
            ylabel=target_name,
            outdir=PLOTS_DIR,
            q2=q2,
            mae_cv=mae_cv,
            suffix=suffix,
        )

        summary_rows.append({
            "Descriptor": SINGLE_DESCRIPTOR,
            "Model": MODEL_NAME,
            "Train_Fraction": float(fraction),
            "Train_Percent": int(round(fraction * 100)),
            "Repeat": int(repeat_idx + 1),
            "Q2(LOGO)": q2,
            "MAE(cv)": mae_cv,
            "RMSE(cv)": rmse_cv,
            "n_samples": int(len(y)),
            "n_groups": int(np.unique(groups).size),
        })

        print(
            f"    pooled metrics -> Q2(LOGO)={q2:.3f}, "
            f"MAE(cv)={mae_cv:.3f}, RMSE(cv)={rmse_cv:.3f}"
        )

    print(
        f"  Fraction {fraction:.0%} summary over {N_SUBSAMPLE_REPEATS} repeats: "
        f"Q2(LOGO) mean={np.mean(repeat_q2):.3f} ± {np.std(repeat_q2, ddof=1) if len(repeat_q2) > 1 else 0.0:.3f}, "
        f"MAE(cv) mean={np.mean(repeat_mae):.3f} ± {np.std(repeat_mae, ddof=1) if len(repeat_mae) > 1 else 0.0:.3f}"
    )

# --------------------------------------------------------------------
# Save detailed and pooled results
# --------------------------------------------------------------------
df_folds = pd.DataFrame(fold_rows)
df_folds.to_csv(PER_FOLD_CSV, index=False)
print(f"\nSaved per-fold LOGO results to: {PER_FOLD_CSV}")

df_logo = pd.DataFrame(summary_rows)
df_logo.to_csv(LOGO_RESULTS_CSV, index=False)
print(f"Saved pooled LOGO results to: {LOGO_RESULTS_CSV}")
print(f"Parity plots stored under: {PLOTS_DIR}")

df_logo_summary = (
    df_logo.groupby(["Train_Fraction", "Train_Percent"], as_index=False)
           .agg(
               Q2_mean=("Q2(LOGO)", "mean"),
               Q2_std=("Q2(LOGO)", "std"),
               MAE_mean=("MAE(cv)", "mean"),
               MAE_std=("MAE(cv)", "std"),
               RMSE_mean=("RMSE(cv)", "mean"),
               RMSE_std=("RMSE(cv)", "std"),
               repeats=("Repeat", "count"),
           )
           .sort_values("Train_Fraction")
)

display(df_logo_summary)


Running LOGO training-size study for descriptor 'dg_ch3' with model GPR[Matern_1.5 + White]


########################################################################################
Training fraction: 1%
########################################################################################
  Repeat 1/10


/Users/robertmayer/miniconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


    pooled metrics -> Q2(LOGO)=0.886, MAE(cv)=0.307, RMSE(cv)=0.403
  Repeat 2/10


/Users/robertmayer/miniconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


    pooled metrics -> Q2(LOGO)=0.858, MAE(cv)=0.352, RMSE(cv)=0.449
  Repeat 3/10


...

/Users/robertmayer/miniconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/robertmayer/miniconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


    pooled metrics -> Q2(LOGO)=0.843, MAE(cv)=0.345, RMSE(cv)=0.472
  Repeat 9/10
    pooled metrics -> Q2(LOGO)=0.822, MAE(cv)=0.356, RMSE(cv)=0.502
  Repeat 10/10
    pooled metrics -> Q2(LOGO)=0.842, MAE(cv)=0.370, RMSE(cv)=0.474
  Fraction 1% summary over 10 repeats: Q2(LOGO) mean=0.859 ± 0.021, MAE(cv) mean=0.341 ± 0.021

########################################################################################
Training fraction: 2%
########################################################################################
  Repeat 1/10
    pooled metrics -> Q2(LOGO)=0.911, MAE(cv)=0.270, RMSE(cv)=0.355
  Repeat 2/10
    pooled metrics -> Q2(LOGO)=0.874, MAE(cv)=0.313, RMSE(cv)=0.422
  Repeat 3/10
    pooled metrics -> Q2(LOGO)=0.910, MAE(cv)=0.274, RMSE(cv)=0.357
  Repeat 4/10
    pooled metrics -> Q2(LOGO)=0.907, MAE(cv)=0.276, RMSE(cv)=0.363
  Repeat 5/10
    pooled metrics -> Q2(LOGO)=0.909, MAE(cv)=0.270, RMSE(cv)=0.360
  Repeat 6/10


/Users/robertmayer/miniconda3/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:442: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


    pooled metrics -> Q2(LOGO)=0.905, MAE(cv)=0.278, RMSE(cv)=0.367
  Repeat 7/10
    pooled metrics -> Q2(LOGO)=0.909, MAE(cv)=0.275, RMSE(cv)=0.359
  Repeat 8/10
    pooled metrics -> Q2(LOGO)=0.879, MAE(cv)=0.311, RMSE(cv)=0.415
  Repeat 9/10
    pooled metrics -> Q2(LOGO)=0.902, MAE(cv)=0.287, RMSE(cv)=0.374
  Repeat 10/10
    pooled metrics -> Q2(LOGO)=0.902, MAE(cv)=0.287, RMSE(cv)=0.372
  Fraction 2% summary over 10 repeats: Q2(LOGO) mean=0.901 ± 0.013, MAE(cv) mean=0.284 ± 0.016

########################################################################################
Training fraction: 3%
########################################################################################
  Repeat 1/10
    pooled metrics -> Q2(LOGO)=0.925, MAE(cv)=0.249, RMSE(cv)=0.327
  Repeat 2/10
    pooled metrics -> Q2(LOGO)=0.916, MAE(cv)=0.255, RMSE(cv)=0.345
  Repeat 3/10
    pooled metrics -> Q2(LOGO)=0.927, MAE(cv)=0.245, RMSE(cv)=0.322
  Repeat 4/10
    pooled metrics -> Q2(LOGO)=0.931, MAE(cv)=0

,Train_Fraction,Train_Percent,Q2_mean,Q2_std,MAE_mean,MAE_std,RMSE_mean,RMSE_std,repeats
0,0.01,1,0.859134,0.020504,0.340622,0.021015,0.446013,0.032661,10
1,0.02,2,0.900838,0.013259,0.283980,0.015825,0.374411,0.024171,10
2,0.03,3,0.918468,0.008119,0.256059,0.012211,0.339759,0.016888,10
3,0.04,4,0.927237,0.006145,0.241656,0.010236,0.321065,0.013614,10
4,0.05,5,0.931387,0.005191,0.233490,0.008111,0.311831,0.011643,10
5,0.10,10,0.941431,0.002906,0.214508,0.004779,0.288206,0.007131,10
6,0.15,15,0.943366,0.003033,0.209626,0.005627,0.283392,0.007601,10
7,0.20,20,0.944758,0.001729,0.206374,0.003022,0.279947,0.004371,10
8,0.30,30,0.944053,0.001800,0.206788,0.002016,0.281726,0.004532,10


### Step 3: Global LOGO summary and training-size dependence

In this step, we analyze how predictive performance depends on the amount of training data used in the LOGO procedure.

Using the pooled LOGO results generated in Step 2, we:

1. **Aggregate performance across repeats**  
For each training fraction, the results from multiple random subsampling repeats are combined. We compute:
   - the mean and standard deviation of **Q²(LOGO)**,  
   - the mean and standard deviation of **MAE(cv)**, and  
   - the mean and standard deviation of **RMSE(cv)**.  

This provides an estimate of model performance at each training size.

2. **Construct performance curves vs training-set size**  
The aggregated metrics are plotted as a function of the training fraction (expressed as % of the available data  per LOGO fold):
   - **Q²(LOGO) vs training fraction** (prediction quality)  
   - **MAE(cv) vs training fraction** (absolute error)  
   - **RMSE(cv) vs training fraction** (squared-error sensitivity)  

Error bars correspond to the standard deviation across repeats and reflect the variability introduced by random subsampling.

3. **Interpret data-efficiency trends**  
These curves allow us to assess:
   - how quickly predictive performance improves with increasing data,  
   - whether performance saturates at higher training fractions, and  
   - how sensitive the model is to reduced training data.

All figures are saved under:  
`analysis_logo_training_fraction/`, and are based on the pooled LOGO summary
CSV generated in Step 2.

In [20]:
from IPython.display import display
from pathlib import Path
import os

# --------------------------------------------------------------------
# Style settings
# --------------------------------------------------------------------
plt.rcParams.update({
    "figure.dpi": 140,
    "font.family": "Arial",
    "axes.grid": False,
})

BLUE = "#1432F5"

ANALYSIS_LOGO_DIR = Path("analysis_logo_training_fraction")
ANALYSIS_LOGO_DIR.mkdir(exist_ok=True, parents=True)

# Load pooled LOGO summary
df_summary = pd.read_csv(SUMMARY_CSV)
df_summary = df_summary.rename(columns={c: c.strip() for c in df_summary.columns})

required_cols = {
    "Train_Fraction", "Train_Percent", "Q2(LOGO)", "MAE(cv)", "RMSE(cv)", "Repeat"
}
missing = required_cols.difference(df_summary.columns)
if missing:
    raise ValueError(f"Missing expected columns in summary CSV: {sorted(missing)}")

curve_df = (
    df_summary.groupby(["Train_Fraction", "Train_Percent"], as_index=False)
              .agg(
                  Q2_mean=("Q2(LOGO)", "mean"),
                  Q2_std=("Q2(LOGO)", "std"),
                  MAE_mean=("MAE(cv)", "mean"),
                  MAE_std=("MAE(cv)", "std"),
                  RMSE_mean=("RMSE(cv)", "mean"),
                  RMSE_std=("RMSE(cv)", "std"),
                  repeats=("Repeat", "count"),
              )
              .sort_values("Train_Fraction")
)

curve_df[["Q2_std", "MAE_std", "RMSE_std"]] = (
    curve_df[["Q2_std", "MAE_std", "RMSE_std"]].fillna(0.0)
)

print(f"Training-size summary for {SINGLE_DESCRIPTOR} with {MODEL_NAME}:")
display(curve_df)

# --------------------------------------------------------------------
# 1) Q²(LOGO)
# --------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(4.5, 3))
ax.errorbar(
    curve_df["Train_Percent"],
    curve_df["Q2_mean"],
    yerr=curve_df["Q2_std"],
    marker="o",
    markersize=6,
    linestyle="-",
    linewidth=1.6,
    capsize=4,
    color=BLUE,
)

ax.set_xlabel("Training data used per LOGO fold (%)")
ax.set_ylabel("Q²(LOGO)")
ax.set_title("Prediction quality vs training-set size")
ax.tick_params(direction="in", top=False, right=False)
ax.grid(False)

plt.tight_layout()
q2_curve_file = ANALYSIS_LOGO_DIR / "q2_vs_training_fraction.png"
fig.savefig(q2_curve_file, dpi=300, bbox_inches="tight", transparent=True)
plt.show()
plt.close(fig)

# --------------------------------------------------------------------
# 2) MAE(cv)
# --------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(4.5, 3))
ax.errorbar(
    curve_df["Train_Percent"],
    curve_df["MAE_mean"],
    yerr=curve_df["MAE_std"],
    marker="o",
    markersize=6,
    linestyle="-",
    linewidth=1.6,
    capsize=4,
    color=BLUE,
)

ax.set_xlabel("Training data used per LOGO fold (%)")
ax.set_ylabel("MAE(cv)")
ax.set_title("MAE vs training-set size")
ax.tick_params(direction="in", top=False, right=False)
ax.grid(False)

plt.tight_layout()
mae_curve_file = ANALYSIS_LOGO_DIR / "mae_vs_training_fraction.png"
fig.savefig(mae_curve_file, dpi=300, bbox_inches="tight", transparent=True)
plt.show()
plt.close(fig)

# --------------------------------------------------------------------
# 3) RMSE(cv)
# --------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(4.5, 3))
ax.errorbar(
    curve_df["Train_Percent"],
    curve_df["RMSE_mean"],
    yerr=curve_df["RMSE_std"],
    marker="o",
    markersize=6,
    linestyle="-",
    linewidth=1.6,
    capsize=4,
    color=BLUE,
)

ax.set_xlabel("Training data used per LOGO fold (%)")
ax.set_ylabel("RMSE(cv)")
ax.set_title("RMSE vs training-set size")
ax.tick_params(direction="in", top=False, right=False)
ax.grid(False)

plt.tight_layout()
rmse_curve_file = ANALYSIS_LOGO_DIR / "rmse_vs_training_fraction.png"
fig.savefig(rmse_curve_file, dpi=300, bbox_inches="tight", transparent=True)
plt.show()
plt.close(fig)

print(f"Saved: {q2_curve_file}")
print(f"Saved: {mae_curve_file}")
print(f"Saved: {rmse_curve_file}")

Training-size summary for dg_ch3 with GPR[Matern_1.5 + White]:


,Train_Fraction,Train_Percent,Q2_mean,Q2_std,MAE_mean,MAE_std,RMSE_mean,RMSE_std,repeats
0,0.01,1,0.859134,0.020504,0.340622,0.021015,0.446013,0.032661,10
1,0.02,2,0.900838,0.013259,0.283980,0.015825,0.374411,0.024171,10
2,0.03,3,0.918468,0.008119,0.256059,0.012211,0.339759,0.016888,10
3,0.04,4,0.927237,0.006145,0.241656,0.010236,0.321065,0.013614,10
4,0.05,5,0.931387,0.005191,0.233490,0.008111,0.311831,0.011643,10
5,0.10,10,0.941431,0.002906,0.214508,0.004779,0.288206,0.007131,10
6,0.15,15,0.943366,0.003033,0.209626,0.005627,0.283392,0.007601,10
7,0.20,20,0.944758,0.001729,0.206374,0.003022,0.279947,0.004371,10
8,0.30,30,0.944053,0.001800,0.206788,0.002016,0.281726,0.004532,10


Saved: analysis_logo_training_fraction/q2_vs_training_fraction.png
Saved: analysis_logo_training_fraction/mae_vs_training_fraction.png
Saved: analysis_logo_training_fraction/rmse_vs_training_fraction.png


/var/folders/1g/tqprlkrd1bx9j8npw4wpqsh00000gn/T/ipykernel_25197/2762945044.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/1g/tqprlkrd1bx9j8npw4wpqsh00000gn/T/ipykernel_25197/2762945044.py:104: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/1g/tqprlkrd1bx9j8npw4wpqsh00000gn/T/ipykernel_25197/2762945044.py:132: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
